# Practice 3 - Get started with Hugging Face
## Exercise 1: Sentiment Analysis with Hugging Face

### 1. Cài đặt thư viện transformers
Đã cài trong Terminal: `python -m pip install torch transformers`. Cell dưới kiểm tra phiên bản.

In [1]:
import torch, transformers
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("GPU:", torch.cuda.is_available())

d:\Code\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch: 2.14.1+cu126
Transformers: 5.21.0
GPU: True


### 2. Dùng model đã huấn luyện sẵn cho phân tích cảm xúc từ Hugging Face Hub

In [2]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
model.eval()
print(model.config.id2label)

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 594.78it/s]

{0: 'NEGATIVE', 1: 'POSITIVE'}


### 3. Tokenize một câu mẫu

In [3]:
sentence = "I really love this movie, it was amazing!"
inputs = tokenizer(sentence, return_tensors="pt")

print("Câu:", sentence)
print("Tokens:", tokenizer.tokenize(sentence))
print("Input IDs:", inputs["input_ids"].tolist())

Câu: I really love this movie, it was amazing!
Tokens: ['i', 'really', 'love', 'this', 'movie', ',', 'it', 'was', 'amazing', '!']
Input IDs: [[101, 1045, 2428, 2293, 2023, 3185, 1010, 2009, 2001, 6429, 999, 102]]


### 4. Phân tích cảm xúc của câu

In [4]:
with torch.no_grad():
    logits = model(**inputs).logits

probs = torch.softmax(logits, dim=1)[0]
label = model.config.id2label[probs.argmax().item()]
print(f"Kết quả: {label} (độ tin cậy {probs.max():.2%})")

Kết quả: POSITIVE (độ tin cậy 99.99%)


### Thử nghiệm thêm với nhiều câu

In [5]:
sentences = [
    "This product is terrible, I want my money back.",
    "The food was okay, nothing special.",
    "Not bad at all!",
    "Oh great, another Monday...",
]
batch = tokenizer(sentences, padding=True, return_tensors="pt")
with torch.no_grad():
    probs = torch.softmax(model(**batch).logits, dim=1)

for s, p in zip(sentences, probs):
    print(f"{model.config.id2label[p.argmax().item()]:8s} {p.max():.2%} | {s}")

NEGATIVE 99.98% | This product is terrible, I want my money back.
NEGATIVE 98.34% | The food was okay, nothing special.
POSITIVE 99.96% | Not bad at all!
POSITIVE 99.70% | Oh great, another Monday...


### Nhận xét
- Model phân loại đúng các câu có cảm xúc rõ ràng và hiểu được phủ định ("Not bad at all!" → POSITIVE).
- Hạn chế 1: model chỉ có 2 nhãn nên câu trung tính ("okay, nothing special") vẫn bị gán NEGATIVE với độ tin cậy rất cao.
- Hạn chế 2: model không nhận ra câu mỉa mai ("Oh great, another Monday..." → POSITIVE).
- Độ tin cậy cao không đồng nghĩa với dự đoán đúng.